# Multi-Layer Yield Curve Network Evolution Analysis

This notebook extends the network evolution framework to multi-layer networks using yield curve zero rates data.
Each layer represents a different maturity term (6M, 1Y, 2Y, 5Y, etc.), with nodes as bond issuers.
Inter-layer edges connect the same issuer across maturity terms, creating a temporal yield curve structure.

## Key Concepts
- **Intra-layer edges**: Correlations between issuers within the same term
- **Inter-layer edges**: Direct connections linking the same issuer across terms
- **Temporal evolution**: Rolling window networks over time for each layer + aggregate multi-layer structure
- **Multi-layer metrics**: Degree centrality, clustering, betweenness accounting for all layers

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import polars as pl
import networkx as nx
from pathlib import Path
import duckdb
from tqdm import tqdm
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
from matplotlib.figure import Figure
import seaborn as sns

# TGraph imports
from tgraphportfolio.analysis.measures import compute_measure, available_measures
from tgraphportfolio.analysis.network import build_corr_nx, pivot_to_wide
from tgraphportfolio.analysis.evolution import (
    EvolutionConfig, CommunityMethod, compute_community_metrics
)

print(f'Imports successful at {datetime.now()}')

## Chapter A: Data Loading and Exploration

Load zero-coupon yield rates from DuckDB and understand the multi-layer structure.

In [ ]:
# Connect to DuckDB
db_path = r'D:\data\duckdb\ycs_data.duckdb'
conn = duckdb.connect(db_path, read_only=True)

# Load zero_rates data
df_raw = conn.execute(
    'SELECT * FROM zero_rates ORDER BY date, source, term'
).pl()

# Clean and explore
df_clean = df_raw.drop_nulls(subset=['date', 'source', 'term', 'rate']).sort(
    ['date', 'source', 'term']
)

dates_unique = sorted(df_clean.select('date').unique()['date'].to_list())
terms_unique = sorted(df_clean.select('term').unique()['term'].to_list())
sources_unique = sorted(df_clean.select('source').unique()['source'].to_list())

print(f'Loaded {len(df_clean)} records')
print(f'Date range: {dates_unique[0]} to {dates_unique[-1]} ({len(dates_unique)} dates)')
print(f'Terms: {len(terms_unique)} unique terms')
print(f'Issuers/Sources: {len(sources_unique)} unique sources')
print(f'\nTerms: {terms_unique}')

## Roadmap for Future Development

- **Omnibus Embedding**: Simultaneous multi-layer embedding via tensor decomposition
- **Tensor Centrality**: Cross-layer influence and spreading dynamics
- **Layer-Aware Modularity**: Community detection preserving layer structure
- **DCC-GARCH**: Time-varying correlations with regime dependence
- **GUI Integration**: Multi-layer tab in TGraph for interactive analysis
- **Causal Inference**: Granger causality on factors
- **Reinforcement Learning**: Optimal rebalancing using network state
- **Anomaly Detection**: Identifying outlier issuers and structural breaks
- **Publication Framework**: Academic-grade analysis and reproducible research